In [11]:
import sqlite3
import pandas as pd

In [12]:
conn = sqlite3.connect("star_schema.db")
cursor = conn.cursor()

In [13]:
df = pd.read_csv("dataset.csv")
df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Postal Code,City,...,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit,Shipping Cost,Order Priority
0,40098,CA-2014-AB10015140-41954,11/11/2014,11/13/2014,First Class,AB-100151402,Aaron Bergman,Consumer,73120.0,Oklahoma City,...,TEC-PH-5816,Technology,Phones,Samsung Convoy 3,221.98,2,0.0,62.15,40.77,High
1,26341,IN-2014-JR162107-41675,2/5/2014,2/7/2014,Second Class,JR-162107,Justin Ritter,Corporate,NaN,Wollongong,...,FUR-CH-5379,Furniture,Chairs,"Novimex Executive Leather Armchair, Black",3709.40,9,0.1,-288.77,923.63,Critical
2,25330,IN-2014-CR127307-41929,10/17/2014,10/18/2014,First Class,CR-127307,Craig Reiter,Consumer,NaN,Brisbane,...,TEC-PH-5356,Technology,Phones,"Nokia Smart Phone, with Caller ID",5175.17,9,0.1,919.97,915.49,Medium
3,13524,ES-2014-KM1637548-41667,1/28/2014,1/30/2014,First Class,KM-1637548,Katherine Murray,Home Office,NaN,Berlin,...,TEC-PH-5267,Technology,Phones,"Motorola Smart Phone, Cordless",2892.51,5,0.1,-96.54,910.16,Medium
4,47221,SG-2014-RH9495111-41948,11/5/2014,11/6/2014,Same Day,RH-9495111,Rick Hansen,Consumer,NaN,Dakar,...,TEC-CO-6011,Technology,Copiers,"Sharp Wireless Fax, High-Speed",2832.96,8,0.0,311.52,903.04,Critical


In [14]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS dim_customer (
    customer_id INTEGER PRIMARY KEY AUTOINCREMENT,
    customer_name TEXT,
    segment TEXT
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS dim_product (
    product_id INTEGER PRIMARY KEY AUTOINCREMENT,
    product_name TEXT,
    category TEXT,
    sub_category TEXT
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS dim_date (
    date_id INTEGER PRIMARY KEY AUTOINCREMENT,
    order_date TEXT,
    year INTEGER,
    month INTEGER,
    day INTEGER
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS dim_region (
    region_id INTEGER PRIMARY KEY AUTOINCREMENT,
    region TEXT,
    country TEXT
)
""")


In [15]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS fact_sales (
    sales_id INTEGER PRIMARY KEY AUTOINCREMENT,
    customer_id INTEGER,
    product_id INTEGER,
    date_id INTEGER,
    region_id INTEGER,
    sales REAL,
    quantity INTEGER,
    profit REAL,
    FOREIGN KEY (customer_id) REFERENCES dim_customer(customer_id),
    FOREIGN KEY (product_id) REFERENCES dim_product(product_id),
    FOREIGN KEY (date_id) REFERENCES dim_date(date_id),
    FOREIGN KEY (region_id) REFERENCES dim_region(region_id)
)
""")


In [16]:
df[['Customer Name','Segment']].drop_duplicates().rename(
    columns={'Customer Name':'customer_name'}
).to_sql('dim_customer', conn, if_exists='append', index=False)

df[['Product Name','Category','Sub-Category']].drop_duplicates().rename(
    columns={'Product Name':'product_name','Sub-Category':'sub_category'}
).to_sql('dim_product', conn, if_exists='append', index=False)

df['Order Date'] = pd.to_datetime(df['Order Date'])
date_df = df[['Order Date']].drop_duplicates()
date_df['year'] = date_df['Order Date'].dt.year
date_df['month'] = date_df['Order Date'].dt.month
date_df['day'] = date_df['Order Date'].dt.day
date_df.rename(columns={'Order Date':'order_date'}).to_sql(
    'dim_date', conn, if_exists='append', index=False
)

df[['Region','Country']].drop_duplicates().rename(
    columns={'Region':'region','Country':'country'}
).to_sql('dim_region', conn, if_exists='append', index=False)


96

In [17]:
df.to_sql("sales_raw", conn, if_exists='replace', index=False)

1000

In [18]:
cursor.execute("""
INSERT INTO fact_sales (
    customer_id, product_id, date_id, region_id,
    sales, quantity, profit
)
SELECT
    c.customer_id,
    p.product_id,
    d.date_id,
    r.region_id,
    s.Sales,
    s.Quantity,
    s.Profit
FROM sales_raw s
JOIN dim_customer c ON s."Customer Name" = c.customer_name
JOIN dim_product p ON s."Product Name" = p.product_name
JOIN dim_date d ON s."Order Date" = d.order_date
JOIN dim_region r ON s.Region = r.region
""")

conn.commit()

In [19]:
cursor.execute("CREATE INDEX IF NOT EXISTS idx_customer ON fact_sales(customer_id)")
cursor.execute("CREATE INDEX IF NOT EXISTS idx_product ON fact_sales(product_id)")
cursor.execute("CREATE INDEX IF NOT EXISTS idx_date ON fact_sales(date_id)")
cursor.execute("CREATE INDEX IF NOT EXISTS idx_region ON fact_sales(region_id)")


In [20]:
query = """
SELECT r.region, SUM(f.sales) AS total_sales
FROM fact_sales f
JOIN dim_region r ON f.region_id = r.region_id
GROUP BY r.region
"""
analysis = pd.read_sql(query, conn)

In [21]:
analysis.to_csv("analysis_outputs.csv", index=False)

In [22]:
sql_file = """
-- TASK 9: STAR SCHEMA SQL
-- Dimension Tables + Fact Table
-- Google Colab + SQLite
"""

with open("task9_star_schema.sql", "w") as f:
    f.write(sql_file)

print(" TASK 9 COMPLETED SUCCESSFULLY")
print("Files generated:")
print("1. star_schema.db")
print("2. analysis_outputs.csv")
print("3. task9_star_schema.sql")

 TASK 9 COMPLETED SUCCESSFULLY
Files generated:
1. star_schema.db
2. analysis_outputs.csv
3. task9_star_schema.sql
